# 🔬 ViTHSD - Dataset A - ViSoBERT (Standard Training)

**Model 3/3 cho Dataset A**

- **Model**: ViSoBERT (`uitnlp/visobert`) — BERT pre-trained trên Vietnamese social media text
- **Dataset**: A (Original ViTHSD - chỉ content + labels)
- **Task**: Multi-Label Classification (11 labels)
- **Technique**: Standard Training (no KD, no CoT)

### 📌 Label Constraint
- Each category can only have **ONE level**: offensive OR hate (not both)

### 🔑 Tại sao ViSoBERT?
- PhoBERT được pre-train trên Wikipedia + báo chí (văn phong trang trọng)
- ViSoBERT được pre-train trên dữ liệu **mạng xã hội** (Facebook, YouTube, TikTok)
- → Phù hợp hơn với ViTHSD — dataset từ mạng xã hội tiếng Việt

## Step 1: Setup Environment

In [1]:
# Cài đặt các thư viện cần thiết
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl

# Fix CUDA - dùng torch >= 2.6 với cu121 (compatible với P100)
!pip install torch==2.6.0 torchvision --index-url https://download.pytorch.org/whl/cu121 -q

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')

print("✅ Dependencies installed!")

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 3.20.0 which is incompatible.
google-cloud-videointelligence 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
google-cloud-vision 3.12.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
ray

In [2]:
# 🎮 GPU Auto-Detection & Optimization
import torch
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"\n{'='*70}")
    print(f"🎮 DETECTED: {num_gpus} GPU(s)")
    print(f"{'='*70}")

    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")

    if num_gpus == 1:
        # Single GPU (P100 16GB)
        print(f"\n{'='*70}")
        print("⚙️  SINGLE GPU MODE (P100 Optimized)")
        print(f"{'='*70}")

        os.environ['CUDA_VISIBLE_DEVICES'] = '0'

        BATCH_SIZE_VISOBERT = 12   # ViSoBERT ~110M params, same footprint as PhoBERT

        GPU_CONFIG = {
            'num_gpus': 1,
            'device': 'cuda:0',
            'batch_sizes': {'visobert': BATCH_SIZE_VISOBERT},
            'gradient_accumulation_steps': 2
        }

    elif num_gpus >= 2:
        # Multi-GPU (T4 x2 32GB)
        print(f"\n{'='*70}")
        print("⚡ MULTI-GPU MODE (T4 x2 Maximum Performance)")
        print(f"{'='*70}")

        BATCH_SIZE_VISOBERT = 16

        GPU_CONFIG = {
            'num_gpus': num_gpus,
            'device': 'cuda',
            'batch_sizes': {'visobert': BATCH_SIZE_VISOBERT},
            'gradient_accumulation_steps': 1
        }

    print(f"\n📊 Configured Batch Size:")
    print(f"   ViSoBERT: {GPU_CONFIG['batch_sizes']['visobert']}")
    print(f"\n{'='*70}\n")

else:
    print("❌ No GPU detected! This notebook requires GPU.")
    GPU_CONFIG = None

PyTorch version: 2.10.0+cu128
CUDA available: True

🎮 DETECTED: 2 GPU(s)
GPU 0: Tesla T4
GPU 1: Tesla T4

⚡ MULTI-GPU MODE (T4 x2 Maximum Performance)

📊 Configured Batch Size:
   ViSoBERT: 16




## Step 2: Import và Setup Paths

In [3]:
import sys
import os

# Copy files ra working directory
!cp -r /kaggle/input/vithsd-experiment/* /kaggle/working/

os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

print("📁 Files copied to working directory")

📁 Files copied to working directory


In [4]:
# Import modules từ vithsd-experiment
from config import FINAL_LABELS, DATA_DIR, OUTPUT_DIR
from data_preparation import load_dataset_A
from evaluation import Evaluator

print("✅ All modules imported!")
print(f"📊 Labels: {len(FINAL_LABELS)}")

✅ All modules imported!
📊 Labels: 11


## Step 3: Load Dataset A

In [5]:
# Load Dataset A
train_texts, train_labels = load_dataset_A('train')
test_texts, test_labels = load_dataset_A('test')

print(f"📊 Dataset A:")
print(f"   Train: {len(train_texts)} samples")
print(f"   Test:  {len(test_texts)} samples")
print(f"   Labels: {FINAL_LABELS}")

[ViHSDLoader] Loaded 7000 samples from train
[ViHSDLoader] Loaded 1800 samples from test
📊 Dataset A:
   Train: 7000 samples
   Test:  1800 samples
   Labels: ['normal', 'individuals#offensive', 'individuals#hate', 'groups#offensive', 'groups#hate', 'religion#offensive', 'religion#hate', 'race#offensive', 'race#hate', 'politics#offensive', 'politics#hate']


## Step 4: Define ViSoBERT Model

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import numpy as np
from tqdm import tqdm

device = torch.device(GPU_CONFIG['device'] if GPU_CONFIG else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"🖥️  Using device: {device}")

VISOBERT_MODEL_ID = 'uitnlp/visobert'
MAX_LEN           = 256
NUM_LABELS        = len(FINAL_LABELS)
DROPOUT           = 0.1

print(f"📊 NUM_LABELS: {NUM_LABELS}")
print(f"🤗 Model: {VISOBERT_MODEL_ID}")

🖥️  Using device: cuda
📊 NUM_LABELS: 11
🤗 Model: uitnlp/visobert


In [7]:
# Load ViSoBERT tokenizer
print("📥 Loading ViSoBERT tokenizer...")
visobert_tokenizer = AutoTokenizer.from_pretrained(VISOBERT_MODEL_ID)
print(f"✅ Tokenizer loaded | Vocab size: {visobert_tokenizer.vocab_size}")

📥 Loading ViSoBERT tokenizer...


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

✅ Tokenizer loaded | Vocab size: 15002


In [8]:
# ── Dataset ──
class ViTHSDDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = np.array(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return (
            enc['input_ids'].squeeze(0),
            enc['attention_mask'].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

print("✅ ViTHSDDataset defined")

✅ ViTHSDDataset defined


In [9]:
# ── ViSoBERT Classifier ──
class ViSoBERTClassifier(nn.Module):
    """
    ViSoBERT encoder + [CLS] pooling + multi-label head.
    Architecture identical to PhoBERT baseline — only backbone changes.
    """
    def __init__(self, model_id=VISOBERT_MODEL_ID, num_labels=NUM_LABELS, dropout=DROPOUT):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_id)
        hidden_size     = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]   # [CLS] token
        logits = self.classifier(self.dropout(pooled))
        return logits

print("✅ ViSoBERTClassifier defined")

✅ ViSoBERTClassifier defined


In [10]:
# ── Wrapper class (giao diện giống PhoBERT notebook) ──
class ViSoBERTModel:
    def __init__(self, dataset_type='A', batch_size=12, num_epochs=3,
                 learning_rate=2e-5, threshold=0.5):
        self.dataset_type = dataset_type
        self.batch_size   = batch_size
        self.num_epochs   = num_epochs
        self.lr           = learning_rate
        self.threshold    = threshold
        self.model        = None
        self.best_state   = None

    def _get_criterion(self, labels_array):
        pos = labels_array.sum(axis=0)
        neg = len(labels_array) - pos
        pw  = torch.tensor(neg / (pos + 1e-6), dtype=torch.float32).to(device)
        return nn.BCEWithLogitsLoss(pos_weight=pw)

    def train(self, train_texts, train_labels):
        train_labels_arr = np.array(train_labels, dtype=np.float32)

        ds_train  = ViTHSDDataset(train_texts, train_labels_arr, visobert_tokenizer)
        loader    = DataLoader(ds_train, batch_size=self.batch_size,
                               shuffle=True, num_workers=2, pin_memory=True)

        print("📥 Loading ViSoBERT encoder weights...")
        self.model = ViSoBERTClassifier().to(device)

        # Multi-GPU support
        if GPU_CONFIG and GPU_CONFIG['num_gpus'] >= 2:
            self.model = nn.DataParallel(self.model)

        criterion   = self._get_criterion(train_labels_arr)
        optimizer   = AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)

        total_steps = len(loader) * self.num_epochs
        scheduler   = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        grad_accum  = GPU_CONFIG['gradient_accumulation_steps']
        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"📊 Trainable parameters: {total_params:,}")

        best_f1 = 0

        for epoch in range(1, self.num_epochs + 1):
            self.model.train()
            total_loss = 0
            optimizer.zero_grad()

            pbar = tqdm(enumerate(loader), total=len(loader),
                        desc=f"Epoch {epoch}/{self.num_epochs}")

            for step, (ids, mask, labels) in pbar:
                ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
                logits = self.model(ids, mask)
                loss   = criterion(logits, labels) / grad_accum
                loss.backward()

                if (step + 1) % grad_accum == 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                total_loss += loss.item() * grad_accum
                pbar.set_postfix({'loss': f'{total_loss/(step+1):.4f}'})

            # Quick train-set F1 for tracking
            preds_train, _ = self._predict_raw(train_texts)
            from sklearn.metrics import f1_score
            f1_mi = f1_score(train_labels_arr, preds_train, average='micro', zero_division=0) * 100
            f1_ma = f1_score(train_labels_arr, preds_train, average='macro', zero_division=0) * 100

            print(f"   Epoch {epoch:02d} | Loss: {total_loss/len(loader):.4f} "
                  f"| Train F1-Mi: {f1_mi:.2f}% | Train F1-Ma: {f1_ma:.2f}%")

            if f1_mi > best_f1:
                best_f1 = f1_mi
                self.best_state = {k: v.cpu().clone()
                                   for k, v in self.model.state_dict().items()}

        if self.best_state:
            self.model.load_state_dict(self.best_state)
        print(f"\n✅ Training complete. Best Train F1-Micro: {best_f1:.2f}%")

    def _predict_raw(self, texts):
        self.model.eval()
        ds     = ViTHSDDataset(texts, np.zeros((len(texts), NUM_LABELS)), visobert_tokenizer)
        loader = DataLoader(ds, batch_size=self.batch_size * 2,
                            shuffle=False, num_workers=2, pin_memory=True)
        all_preds, all_probs = [], []
        with torch.no_grad():
            for ids, mask, _ in loader:
                ids, mask = ids.to(device), mask.to(device)
                logits    = self.model(ids, mask)
                probs     = torch.sigmoid(logits).cpu().numpy()
                preds     = (probs > self.threshold).astype(np.float32)
                preds     = self._enforce_constraint(preds)
                all_preds.append(preds)
                all_probs.append(probs)
        return np.vstack(all_preds), np.vstack(all_probs)

    def _enforce_constraint(self, preds_batch):
        """
        One-level-per-category: nếu cả Offensive + Hate cùng được predict → giữ Hate.
        """
        TARGETS   = ['Individuals', 'Groups', 'Race', 'Politics', 'Religion']
        label2idx = {l: i for i, l in enumerate(FINAL_LABELS)}
        for row in preds_batch:
            for t in TARGETS:
                off_key  = f'{t}#Offensive'
                hate_key = f'{t}#Hate'
                if off_key in label2idx and hate_key in label2idx:
                    off_i  = label2idx[off_key]
                    hate_i = label2idx[hate_key]
                    if row[off_i] == 1.0 and row[hate_i] == 1.0:
                        row[off_i] = 0.0
        return preds_batch

    def predict(self, texts):
        return self._predict_raw(texts)

print("✅ ViSoBERTModel wrapper defined")

✅ ViSoBERTModel wrapper defined


## Step 5: Train ViSoBERT

In [11]:
from models import convert_to_binary

train_labels_bin = convert_to_binary(train_labels)
test_labels_bin  = convert_to_binary(test_labels)

In [12]:
# Initialize
evaluator = Evaluator()

# ViSoBERT - Dataset A (Standard Training)
print("="*70)
print("🚀 ViSoBERT - DATASET A (Standard)")
print("="*70)

model_visobert = ViSoBERTModel(
    dataset_type='A',
    batch_size=GPU_CONFIG['batch_sizes']['visobert'],  # Auto-adjusted
    num_epochs=3,
    learning_rate=2e-5
)

# Train
model_visobert.train(train_texts, train_labels_bin)

# Predict
pred_visobert, _ = model_visobert.predict(test_texts)

# Evaluate
result = evaluator.evaluate(test_labels, pred_visobert, "ViSoBERT", "A")
evaluator.print_result(result)

🚀 ViSoBERT - DATASET A (Standard)
📥 Loading ViSoBERT encoder weights...


pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

   Epoch 01 | Loss: 1.3849 | Train F1-Mi: 45.22% | Train F1-Ma: 31.16%


Epoch 2/3: 100%|██████████| 438/438 [03:17<00:00,  2.22it/s, loss=1.0355]


   Epoch 02 | Loss: 1.0355 | Train F1-Mi: 58.44% | Train F1-Ma: 49.40%


Epoch 3/3: 100%|██████████| 438/438 [03:17<00:00,  2.21it/s, loss=0.6604]


   Epoch 03 | Loss: 0.6604 | Train F1-Mi: 66.57% | Train F1-Ma: 55.41%

✅ Training complete. Best Train F1-Micro: 66.57%

📊 ViSoBERT on Dataset A (Multi-Label)
  Subset Accuracy (Exact Match): 0.4289
  Hamming Loss:                  0.1273
  
  F1 Samples:                    0.6205
  F1 Macro:                      0.4144
  F1 Micro:                      0.5569
  
  Precision (Samples):           0.5797
  Precision (Macro):             0.3325
  Precision (Micro):             0.4579
  
  Recall (Samples):              0.7344
  Recall (Macro):                0.6146
  Recall (Micro):                0.7103

  F1 per Label:
    normal                   : 0.7821
    individuals#offensive    : 0.4783
    individuals#hate         : 0.5937
    groups#offensive         : 0.3472
    groups#hate              : 0.4604
    religion#offensive       : 0.8000
    race#offensive           : 0.2188
    race#hate                : 0.2979
    politics#offensive       : 0.1463
    politics#hate            : 0

## Step 6: Save Results

In [13]:
import json
from pathlib import Path
import numpy as np

# Create output directory
output_dir = Path("/kaggle/working/outputs")
output_dir.mkdir(exist_ok=True)

# Convert numpy types
def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Save
results = {'ViSoBERT': convert_to_serializable(result)}
with open(output_dir / "results_datasetA_visobert.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Results saved to /kaggle/working/outputs/results_datasetA_visobert.json")
print(f"\n📊 ViSoBERT Results:")
print(f"   Subset Accuracy: {result['subset_accuracy']:.4f}")
print(f"   Macro F1: {result['f1_macro']:.4f}")
print(f"   Micro F1: {result['f1_micro']:.4f}")

✅ Results saved to /kaggle/working/outputs/results_datasetA_visobert.json

📊 ViSoBERT Results:
   Subset Accuracy: 0.4289
   Macro F1: 0.4144
   Micro F1: 0.5569


## Step 7: Save Model State Dict

Lưu state_dict để có thể load lại model sau này với đầy đủ attention weights (cho hate word extraction)

In [14]:
import torch
from pathlib import Path

# Create model checkpoint directory
checkpoint_dir = Path("/kaggle/working/checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

# Save state_dict
checkpoint_path = checkpoint_dir / "visobert_datasetA_state_dict.pt"

torch.save({
    'model_state_dict': model_visobert.model.state_dict(),
    'model_config': {
        'model_name': 'uitnlp/visobert',
        'num_labels': NUM_LABELS,
        'problem_type': 'multi_label_classification'
    },
    'training_info': {
        'dataset': 'A',
        'technique': 'Standard Training',
        'subset_accuracy': result['subset_accuracy'],
        'f1_macro': result['f1_macro'],
        'f1_micro': result['f1_micro']
    }
}, checkpoint_path)

print(f"✅ State dict saved to: {checkpoint_path}")
print(f"📦 File size: {checkpoint_path.stat().st_size / (1024**2):.2f} MB")
print("\n💡 Cách load lại:")
print("   checkpoint = torch.load('visobert_datasetA_state_dict.pt')")
print("   model = ViSoBERTClassifier()")
print("   model.load_state_dict(checkpoint['model_state_dict'])")
print("   # Sau đó có thể dùng model để extract attention weights")

✅ State dict saved to: /kaggle/working/checkpoints/visobert_datasetA_state_dict.pt
📦 File size: 372.30 MB

💡 Cách load lại:
   checkpoint = torch.load('visobert_datasetA_state_dict.pt')
   model = ViSoBERTClassifier()
   model.load_state_dict(checkpoint['model_state_dict'])
   # Sau đó có thể dùng model để extract attention weights
